In [1]:
# Importa a função de pontuação de textos por assunto (TF-IDF) e o lematizador de conjuntos
from funcoes_de_analise.catalogar_textos_longos import catalogar_textos_longos, lematizar_conjunto

In [2]:
# Carrega o corpus (dados_v02.pickle) e monta a lista de todas as sentenças
# CARREGAR ARQUIVO PICKLE (OBRIGATÓRIO)
# Carrega as principais variáveis já processadas, útil para economizar tempo
import pickle
caminho_arquivo = 'corpora/dados_v02.pickle'
with open(caminho_arquivo, 'rb') as arquivo_entrada:
   transcricoes_dados_dict = pickle.load(arquivo_entrada)
del arquivo_entrada, caminho_arquivo

# Cria a lista de sentenças
sentencas = [sentenca for transcricao in transcricoes_dados_dict for sentenca in transcricao["lista_de_sentencas"]]

In [3]:
# Extrai os títulos (identificadores) e os textos (transcrições) de cada vídeo
titulos = [titulo["titulo"]for titulo in transcricoes_dados_dict]
textos = [texto["transcricao"] for texto in transcricoes_dados_dict]

In [4]:
# Define os conjuntos de palavras-chave que caracterizam cada tema/assunto
religiao = {"deus", "senhor", "oração", "jesus", "igreja"}
atuacao_politica = {"ministro", "presidente", "deputado", "comissão", "bolsonaro"}
kairos = {"dia", "ano", "tempo", "momento", "hora"}
familia = {"pai", "criança", "casa", "família", "filho"}
patria = {"brasil", "país", "mundo", "nação", }
esquerda = {"lula", "homem", "esquerda", "mulher", "pt"}
verdade = {"verdade", "fato", "questão"}
educacao = {"educação", "escola", "professor","universidade", "aula "}

In [5]:
# Pontua (TF-IDF) cada texto em relação a cada tema; os sets de palavras são lematizados na chamada
religiao = catalogar_textos_longos(titulos, textos, lematizar_conjunto(religiao), "Religião")
atuacao_politica = catalogar_textos_longos(titulos, textos, lematizar_conjunto(atuacao_politica), "Atuação Política")
kairos = catalogar_textos_longos(titulos, textos, lematizar_conjunto(kairos), "Kairós")
familia = catalogar_textos_longos(titulos, textos, lematizar_conjunto(familia), "Família")
patria = catalogar_textos_longos(titulos, textos, lematizar_conjunto(patria), "Pátria")
esquerda = catalogar_textos_longos(titulos, textos, lematizar_conjunto(esquerda), "Esquerda")
verdade = catalogar_textos_longos(titulos, textos, lematizar_conjunto(verdade), "Verdade")
educacao = catalogar_textos_longos(titulos, textos, lematizar_conjunto(educacao), "Educação")


Lematizando corpus: 99.4% [472/475]


In [6]:
# Consolida os temas: para cada título, une as grandezas de todos os temas num único dict {titulo: {todos os temas}}
dicionario_final = {}

for chave, valor in religiao.items():
    dicionario_final[chave] = valor | atuacao_politica[chave] | kairos[chave] | familia[chave] | patria[chave] | esquerda[chave] | verdade[chave] | educacao[chave]


In [7]:
# Calcula a correlação entre os temas e salva o heatmap (Pearson vs Spearman); imprime o resumo por tema
from funcoes_de_analise.correlacao_entre_temas import correlacao_entre_temas

corr, resumo = correlacao_entre_temas(
   dicionario_final,
    metodo="spearman",                              # recomendado p/ TF-IDF esparso
    caminho_heatmap="graficos/correlacao_temas.html"
)
print(resumo.to_string(index=False))


Heatmap (Pearson vs Spearman) salvo em: graficos/correlacao_temas.html
            tema  n_docs_positivos pct_docs_positivos mais_correlacionado  corr_max mais_anticorrelacionado  corr_min  isolamento
        Educação               189              40.0%             Família     0.263        Atuação Política     0.055       0.129
         Verdade               364              77.1%    Atuação Política     0.280                 Família     0.100       0.162
        Esquerda               351              74.4%              Pátria     0.341                 Família     0.060       0.176
         Família               363              76.9%              Kairós     0.337        Atuação Política    -0.009       0.179
Atuação Política               327              69.3%            Esquerda     0.341                 Família    -0.009       0.183
        Religião               326              69.1%              Kairós     0.341                Educação     0.084       0.198
          Kairós   

In [8]:
import pandas as pd

df = pd.DataFrame.from_dict(dicionario_final, orient="index")
df

,Religião,Atuação Política,Kairós,Família,Pátria,Esquerda,Verdade,Educação
Dia 21/21 - Oração pelo Brasil - Edésio de Oliveira e Rodrigo Aldeia,0.883904,0.013943,0.126834,0.266488,0.102193,0.028286,0.014438,0.002405
NIKOLAS FERREIRA - VIGÍLIA DE ENCERRAMENTO - 21 Dias de Oração pelo Brasil,0.806854,0.018553,0.095817,0.197780,0.168820,0.014850,0.009748,0.002238
Dia 13/21 - Oração pelo Brasil - Cassiane e Elizeu Rodrigues,0.702295,0.000000,0.133589,0.065975,0.048789,0.027594,0.020758,0.004061
"NIKOLAS, JULIO CESAR E DAVI SILVA - 21 Dias de Oração pelo Brasil",0.690688,0.004042,0.101105,0.177870,0.106739,0.064004,0.034209,0.000000
"Dia 10/21 - Oração pelo Brasil - Eyshila, Michelle e Bolsonaro",0.667196,0.006996,0.166883,0.098507,0.098425,0.019366,0.011816,0.000000
...,...,...,...,...,...,...,...,...
EXPONDO TODA A HIPOCRISIA DA ESQUERDA,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Desmascarando PETISTA em minutos,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
NIKOLAS RASGA O VERBO E MANDA RECADO PRA TODO O BRASIL,0.000000,0.204310,0.098791,0.109397,0.202331,0.058850,0.045531,0.015999
Não existe doutrinação?! Tem certeza?,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [13]:
df.to_csv("videos_tf_idf.csv", index=True,  decimal=",", sep=";", float_format="%.3f", encoding="utf-8-sig")

In [14]:
temas_alvo = ["Religião", "Família", "Pátria"]

# 1. Converte cada tema para percentil (0 a 1), deixando as escalas comparáveis
ranks = df[temas_alvo].rank(pct=True)

# 2. Combina exigindo que os TRÊS estejam altos ao mesmo tempo (gargalo = o menor dos três)
df["score_conjunto"] = ranks.min(axis=1)

# 3. Ordena e vê a amostra
amostra = df.sort_values("score_conjunto", ascending=False)
amostra[temas_alvo + ["score_conjunto"]].head(10)


,Religião,Família,Pátria,score_conjunto
NIKOLAS FERREIRA - VIGÍLIA DE ENCERRAMENTO - 21 Dias de Oração pelo Brasil,0.806854,0.197780,0.168820,0.957627
Dia 11/21 - Oração pelo Brasil - David Quinlan e Andre Valadao,0.516230,0.126873,0.137878,0.889831
"NIKOLAS, LUMA ELPIDIO E PAULO BORGES JR - 21 Dias de Oração pelo Brasil",0.383585,0.261575,0.111555,0.866525
"NIKOLAS, JULIO CESAR E DAVI SILVA - 21 Dias de Oração pelo Brasil",0.690688,0.177870,0.106739,0.855932
Dia 03/21 - Oração pelo Brasil - Gabriela Lopes e Arthur Callazans,0.579895,0.148114,0.106281,0.849576
Dia 21/21 - Oração pelo Brasil - Edésio de Oliveira e Rodrigo Aldeia,0.883904,0.266488,0.102193,0.836864
Dia 19/21 - Oração pelo Brasil - Guilherme Batista e David Miranda,0.640278,0.108513,0.100934,0.834746
"NIKOLAS, HANANIEL EDUARDO E JULIO VERTULHO - 21 Dias de Oração pelo Brasil",0.517019,0.099015,0.176067,0.834746
"Dia 10/21 - Oração pelo Brasil - Eyshila, Michelle e Bolsonaro",0.667196,0.098507,0.098425,0.830508
NIKOLAS E SANDRA ALVES - 21 Dias de Oração pelo Brasil,0.362961,0.093867,0.136297,0.815678


In [15]:
amostra.index[0]

'NIKOLAS FERREIRA - VIGÍLIA DE ENCERRAMENTO - 21 Dias de Oração pelo Brasil'